In [0]:
%python
# ============================================================================
# IMPORT PROFESSIONAL EXTRACTION UTILITIES
# ============================================================================

import sys

# Add project paths to sys.path
project_root = "/Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import organization classification utilities
HAS_ORG_CLASSIFICATION = False
HAS_STRUCTURED_MODELS = False

try:
    from prompts_and_pydantic_models.organization_extraction import (
        ORGANIZATION_EXTRACTION_SYSTEM_PROMPT,
        OrganizationExtractionOutput
    )
    print("✓ Imported: Organization classification (NGOs vs Facilities)")
    HAS_ORG_CLASSIFICATION = True
except ImportError as e:
    print(f"⚠️  Organization classification unavailable: {str(e)[:60]}...")

try:
    from prompts_and_pydantic_models.facility_and_ngo_fields import (
        ORGANIZATION_INFORMATION_SYSTEM_PROMPT,
        Facility,
        NGO,
        BaseOrganization
    )
    print("✓ Imported: Structured field models (Facility & NGO)")
    HAS_STRUCTURED_MODELS = True
except ImportError as e:
    print(f"⚠️  Structured models unavailable: {str(e)[:60]}...")

if HAS_ORG_CLASSIFICATION or HAS_STRUCTURED_MODELS:
    print("\n✅ Professional utilities available for silver transformation")
    print("   → Enhanced organization classification")
    print("   → Structured field validation")
else:
    print("\n⚠️  Professional utilities not available - using standard transformation")

In [0]:
%python
# ============================================================================
# STANDARDIZATION LAYER - MEDICAL TERMINOLOGY MAPPINGS
# ============================================================================
# CRITICAL: Normalize free-text medical terms to standard taxonomy
# This ensures consistent data quality across the pipeline

from pyspark.sql import functions as F

# Medical Specialty Standardization
SPECIALTY_MAPPING = {
    # Surgery variations
    "surgery": "generalSurgery",
    "surgical": "generalSurgery",
    "operation": "generalSurgery",
    "operative care": "generalSurgery",
    "general surgery": "generalSurgery",
    "orthopedic": "orthopedicSurgery",
    "orthopaedic": "orthopedicSurgery",
    "cardiac surgery": "cardiacSurgery",
    "heart surgery": "cardiacSurgery",
    "plastic surgery": "plasticSurgery",
    
    # Emergency variations
    "emergency": "emergencyMedicine",
    "emergency care": "emergencyMedicine",
    "er": "emergencyMedicine",
    "ed": "emergencyMedicine",
    "emergency department": "emergencyMedicine",
    "trauma": "criticalCareMedicine",
    "critical care": "criticalCareMedicine",
    "icu": "criticalCareMedicine",
    "intensive care": "criticalCareMedicine",
    
    # Maternal/Child variations
    "maternity": "gynecologyAndObstetrics",
    "obstetric": "gynecologyAndObstetrics",
    "obstetrics": "gynecologyAndObstetrics",
    "gynecology": "gynecologyAndObstetrics",
    "gynaecology": "gynecologyAndObstetrics",
    "women's health": "gynecologyAndObstetrics",
    "pediatric": "pediatrics",
    "paediatric": "pediatrics",
    "children": "pediatrics",
    "neonatal": "neonatologyPerinatalMedicine",
    
    # Diagnostic variations
    "radiology": "radiology",
    "imaging": "radiology",
    "x-ray": "radiology",
    "laboratory": "pathology",
    "lab": "pathology",
    "pathology": "pathology",
    "diagnostic": "pathology",
    
    # Specialty variations
    "eye": "ophthalmology",
    "ophthalmology": "ophthalmology",
    "ophthalmic": "ophthalmology",
    "vision": "ophthalmology",
    "dental": "dentistry",
    "dentistry": "dentistry",
    "teeth": "dentistry",
    "orthodontic": "orthodontics",
    "heart": "cardiology",
    "cardiology": "cardiology",
    "skin": "dermatology",
    "dermatology": "dermatology",
    "ent": "otolaryngology",
    "ear nose throat": "otolaryngology",
    
    # General medicine
    "medical": "internalMedicine",
    "medicine": "internalMedicine",
    "internal medicine": "internalMedicine",
    "general practice": "familyMedicine",
    "family medicine": "familyMedicine",
    "primary care": "familyMedicine"
}

# Procedure Standardization
PROCEDURE_MAPPING = {
    "operation": "surgical_procedure",
    "operations": "surgical_procedure",
    "surgical procedure": "surgical_procedure",
    "surgery": "surgical_procedure",
    "c-section": "cesarean_section",
    "caesarean": "cesarean_section",
    "cesarean section": "cesarean_section",
    "delivery": "childbirth",
    "childbirth": "childbirth",
    "vaccination": "immunization",
    "immunization": "immunization",
    "vaccine": "immunization",
    "consultation": "medical_consultation",
    "checkup": "medical_consultation",
    "check-up": "medical_consultation",
    "diagnosis": "diagnostic_evaluation",
    "diagnostic test": "diagnostic_evaluation"
}

# Equipment Standardization
EQUIPMENT_MAPPING = {
    "xray": "x-ray_machine",
    "x ray": "x-ray_machine",
    "x-ray": "x-ray_machine",
    "mri": "mri_scanner",
    "ct scan": "ct_scanner",
    "cat scan": "ct_scanner",
    "ultrasound": "ultrasound_machine",
    "sonography": "ultrasound_machine",
    "ecg": "ecg_machine",
    "ekg": "ecg_machine",
    "electrocardiogram": "ecg_machine"
}

# Capability Standardization
CAPABILITY_MAPPING = {
    "emergency services": "emergency_care",
    "emergency care": "emergency_care",
    "er services": "emergency_care",
    "trauma care": "emergency_care",
    "24/7": "24_hour_service",
    "24 hour": "24_hour_service",
    "round the clock": "24_hour_service",
    "intensive care": "icu_services",
    "icu": "icu_services",
    "critical care": "icu_services",
    "maternity services": "maternal_care",
    "maternity care": "maternal_care",
    "labor and delivery": "maternal_care",
    "surgical services": "surgical_capability",
    "operating room": "surgical_capability",
    "surgery": "surgical_capability"
}

print("✓ Standardization mappings loaded:")
print(f"  - Specialties: {len(SPECIALTY_MAPPING)} terms")
print(f"  - Procedures: {len(PROCEDURE_MAPPING)} terms")
print(f"  - Equipment: {len(EQUIPMENT_MAPPING)} terms")
print(f"  - Capabilities: {len(CAPABILITY_MAPPING)} terms")
print("\n⚠️  These will be applied during transformation to normalize free text")

In [0]:
%python
# Databricks notebook source
from pyspark.sql import functions as F, types as T
from typing import List

CATALOG = "vf_health"
SCHEMA = "ghana"
BRONZE = f"{CATALOG}.{SCHEMA}.bronze_raw_facilities"
SILVER = f"{CATALOG}.{SCHEMA}.silver_facilities_clean"

df = spark.table(BRONZE)

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

# Parse JSON-like array string columns safely
def parse_array(colname: str):
    raw = F.coalesce(F.col(colname), F.lit("[]"))
    raw = F.when(F.lower(F.trim(raw)).isin("null", ""), F.lit("[]")).otherwise(raw)
    arr = F.from_json(raw, T.ArrayType(T.StringType()))
    return F.coalesce(arr, F.array().cast("array<string>"))

# Create standardization UDF
def make_standardization_udf(mapping: dict):
    """Create a UDF that applies standardization mapping to array elements."""
    def standardize_values(values: List[str]) -> List[str]:
        if not values:
            return []
        
        result = []
        seen = set()
        
        for val in values:
            if not val:
                continue
            
            # Clean the value
            clean_val = val.strip().lower()
            
            # Apply mapping if exists
            standardized = mapping.get(clean_val, clean_val)
            
            # Deduplicate
            if standardized not in seen and standardized:
                seen.add(standardized)
                result.append(standardized)
        
        return result
    
    return F.udf(standardize_values, T.ArrayType(T.StringType()))

# Create UDFs for each mapping
standardize_specialties_udf = make_standardization_udf(SPECIALTY_MAPPING)
standardize_procedure_udf = make_standardization_udf(PROCEDURE_MAPPING)
standardize_equipment_udf = make_standardization_udf(EQUIPMENT_MAPPING)
standardize_capability_udf = make_standardization_udf(CAPABILITY_MAPPING)

# ============================================================================
# DATA CLEANING & VALIDATION
# ============================================================================

total_before = df.count()

clean = (
    df
    # VALIDATION RULE 1: Name must be present
    .filter(F.col("name").isNotNull() & (F.trim(F.col("name")) != ""))
    
    # VALIDATION RULE 2: Email format validation (if provided)
    .withColumn(
        "email_valid",
        F.when(
            F.col("email").isNotNull(),
            F.col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")
        ).otherwise(F.lit(True))  # NULL emails are considered valid
    )
    .withColumn(
        "email",
        F.when(F.col("email_valid"), F.col("email")).otherwise(F.lit(None))
    )
    .drop("email_valid")
    
    # Basic field cleaning
    .withColumn("name", F.trim("name"))
    .withColumn("organization_type", F.lower(F.trim("organization_type")))
    
    # Parse array fields
    .withColumn("specialties", parse_array("specialties"))
    .withColumn("procedure", parse_array("procedure"))
    .withColumn("equipment", parse_array("equipment"))
    .withColumn("capability", parse_array("capability"))
    .withColumn("phone_numbers", parse_array("phone_numbers"))
    .withColumn("websites", parse_array("websites"))
    .withColumn("countries", parse_array("countries"))
    .withColumn("affiliationTypeIds", parse_array("affiliationTypeIds"))
    
    # STANDARDIZATION LAYER: Apply mappings using UDFs
    .withColumn("specialties", standardize_specialties_udf(F.col("specialties")))
    .withColumn("procedure", standardize_procedure_udf(F.col("procedure")))
    .withColumn("equipment", standardize_equipment_udf(F.col("equipment")))
    .withColumn("capability", standardize_capability_udf(F.col("capability")))
    
    # Address defaults
    .withColumn("address_country", F.coalesce(F.col("address_country"), F.lit("Ghana")))
    .withColumn("address_countryCode", F.coalesce(F.col("address_countryCode"), F.lit("GH")))
    
    # Year established handling
    .withColumn("yearEstablished",F.when(F.lower(F.trim(F.coalesce(F.col("yearEstablished"), F.lit("")))).isin("null", ""), None).otherwise(F.col("yearEstablished")).cast("int"))
    
    # Boolean conversion
    .withColumn(
        "acceptsVolunteers",
        F.when(F.lower(F.col("acceptsVolunteers")) == "true", F.lit(True))
         .when(F.lower(F.col("acceptsVolunteers")) == "false", F.lit(False))
         .otherwise(F.lit(None).cast("boolean"))
    )
    
    # VALIDATION RULE 3: Numeric field sanity checks
    .withColumn("area", 
        F.when(F.lower(F.trim(F.coalesce(F.col("area"), F.lit("")))).isin("null", ""), None)
        .otherwise(F.col("area")).cast("int")
    )
    
    # VALIDATION RULE 4: numberDoctors range check (>= 0 AND < 500)
    .withColumn("numberDoctors", 
        F.when(F.lower(F.trim(F.coalesce(F.col("numberDoctors"), F.lit("")))).isin("null", ""), None)
        .otherwise(F.col("numberDoctors")).cast("int")
    )
    .withColumn(
        "numberDoctors",
        F.when(
            (F.col("numberDoctors").isNotNull()) & 
            ((F.col("numberDoctors") < 0) | (F.col("numberDoctors") >= 500)),
            F.lit(None)  # Set invalid values to NULL
        ).otherwise(F.col("numberDoctors"))
    )
    
    # VALIDATION RULE 5: capacity range check (>= 0 AND < 5000)
    .withColumn("capacity", 
        F.when(F.lower(F.trim(F.coalesce(F.col("capacity"), F.lit("")))).isin("null", ""), None)
        .otherwise(F.col("capacity")).cast("int")
    )
    .withColumn(
        "capacity",
        F.when(
            (F.col("capacity").isNotNull()) & 
            ((F.col("capacity") < 0) | (F.col("capacity") >= 5000)),
            F.lit(None)  # Set invalid values to NULL
        ).otherwise(F.col("capacity"))
    )
    
    # Generate row_id and deduplicate
    .withColumn(
        "row_id",
        F.sha2(F.concat_ws("||", F.coalesce(F.col("unique_id"), F.lit("")), F.coalesce(F.col("source_url"), F.lit(""))), 256)
    )
    .dropDuplicates(["unique_id", "source_url"])
    .withColumn("cleaned_at", F.current_timestamp())
)

# ============================================================================
# SELECT FINAL SCHEMA & SAVE
# ============================================================================

silver_df = clean.select(
    "row_id",
    "unique_id",
    "source_url",
    "name",
    "organization_type",
    "specialties",
    "procedure",
    "equipment",
    "capability",
    "phone_numbers",
    "email",
    "websites",
    "officialWebsite",
    "yearEstablished",
    "acceptsVolunteers",
    "facebookLink",
    "twitterLink",
    "linkedinLink",
    "instagramLink",
    "logo",
    "address_line1",
    "address_line2",
    "address_line3",
    "address_city",
    "address_stateOrRegion",
    "address_zipOrPostcode",
    "address_country",
    "address_countryCode",
    "countries",
    "missionStatement",
    "missionStatementLink",
    "organizationDescription",
    "facilityTypeId",
    "operatorTypeId",
    "affiliationTypeIds",
    "description",
    "area",
    "numberDoctors",
    "capacity",
    "cleaned_at"
)

silver_df.write.mode("overwrite").format("delta").saveAsTable(SILVER)

final_count = spark.table(SILVER).count()

# ============================================================================
# VALIDATION & STANDARDIZATION SUMMARY
# ============================================================================

print("\n" + "="*70)
print("DATA QUALITY SUMMARY")
print("="*70)
print(f"Records before transformation: {total_before}")
print(f"Records after transformation: {final_count}")
print(f"Records removed (empty names): {total_before - final_count}")

print("\n✅ VALIDATION RULES APPLIED:")
print("  1. Name validation: Non-null, non-empty")
print("  2. Email validation: Valid format (regex)")
print("  3. numberDoctors range: 0 <= value < 500")
print("  4. capacity range: 0 <= value < 5000")

print("\n✅ STANDARDIZATION APPLIED:")
print("  - Specialties: 56 term mappings (e.g., 'surgery' → 'generalSurgery')")
print("  - Procedures: 17 term mappings (e.g., 'operation' → 'surgical_procedure')")
print("  - Equipment: 11 term mappings (e.g., 'xray' → 'x-ray_machine')")
print("  - Capabilities: 16 term mappings (e.g., '24/7' → '24_hour_service')")

print("\n✅ DATA QUALITY IMPROVEMENTS:")
print("  - Duplicate array elements removed")
print("  - Empty strings filtered from arrays")
print("  - Consistent lowercase normalization")
print("="*70)

print(f"\n✓ Silver table saved: {SILVER}")
print(f"  Final record count: {final_count}")

In [0]:
%python
# ============================================================================
# ENHANCED SILVER LAYER - ADD COMPUTED INTELLIGENCE FIELDS
# ============================================================================
# Add capability detection flags and scoring for medical desert identification

from pyspark.sql import functions as F

print("\n" + "="*70)
print("ENHANCING SILVER TABLE WITH INTELLIGENCE FIELDS")
print("="*70)

# Read the cleaned silver table
silver_base = spark.table(SILVER)

# ============================================================================
# COMPUTED BOOLEAN FLAGS
# ============================================================================

enhanced = (
    silver_base
    
    # 1. HAS_EMERGENCY: Detect emergency/trauma/ICU capabilities
    .withColumn(
        "has_emergency",
        F.expr(
            """exists(capability, x -> lower(x) rlike 'emergency|trauma|\\bicu\\b|nicu|intensive.*care|critical.*care') 
            OR exists(equipment, x -> lower(x) rlike 'emergency|trauma|\\bicu\\b|ventilator')
            OR exists(specialties, x -> lower(x) rlike 'emergencymedicine|criticalcaremedicine')
            """
        ).cast("boolean")
    )
    
    # 2. HAS_SURGERY: Detect surgical capabilities
    .withColumn(
        "has_surgery",
        F.expr(
            """exists(procedure, x -> lower(x) rlike 'surgery|surgical|operation') 
            OR exists(capability, x -> lower(x) rlike 'surgery|surgical|operation|operating.*room|\\bor\\b') 
            OR exists(equipment, x -> lower(x) rlike 'surgical|operating.*room')
            OR exists(specialties, x -> lower(x) rlike 'surgery')
            """
        ).cast("boolean")
    )
    
    # 3. HAS_IMAGING: Detect imaging/radiology capabilities  
    .withColumn(
        "has_imaging",
        F.expr(
            """exists(equipment, x -> lower(x) rlike '\\bmri\\b|\\bct\\b|x-ray|xray|ultrasound|imaging|scanner') 
            OR exists(procedure, x -> lower(x) rlike 'imaging|radiology|\\bmri\\b|\\bct\\b|x-ray|ultrasound')
            OR exists(specialties, x -> lower(x) rlike 'radiology')
            """
        ).cast("boolean")
    )
)

# ============================================================================
# CAPABILITY SCORE (0-8 points)
# ============================================================================
# Scoring system to quantify facility capability level
# Higher score = better resourced facility

enhanced = (
    enhanced
    .withColumn(
        "capability_score",
        (
            # Emergency capability: 2 points
            F.when(F.col("has_emergency"), 2).otherwise(0) +
            
            # Surgical capability: 2 points
            F.when(F.col("has_surgery"), 2).otherwise(0) +
            
            # Imaging capability: 1 point
            F.when(F.col("has_imaging"), 1).otherwise(0) +
            
            # Adequate medical staffing (>= 5 doctors): 2 points
            F.when(F.col("numberDoctors") >= 5, 2).otherwise(0) +
            
            # Adequate bed capacity (>= 20 beds): 1 point
            F.when(F.col("capacity") >= 20, 1).otherwise(0)
        ).cast("int")
    )
)

# ============================================================================
# MEDICAL DESERT RISK FLAG
# ============================================================================
# Facilities with score <= 2 are at risk / minimal capability

enhanced = enhanced.withColumn(
    "is_medical_desert_risk",
    (F.col("capability_score") <= 2).cast("boolean")
)

# ============================================================================
# ADDRESS REGION NORMALIZATION
# ============================================================================
# Create a standardized region field with fallback logic

enhanced = enhanced.withColumn(
    "address_region_clean",
    F.coalesce(
        F.col("address_stateOrRegion"),
        F.col("address_city"),
        F.lit("Unknown")
    )
)

# ============================================================================
# WRITE ENHANCED SILVER TABLE
# ============================================================================

enhanced.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(SILVER)

final_enhanced = spark.table(SILVER)

print("\n✅ INTELLIGENCE FIELDS ADDED:")
print("  1. has_emergency (boolean) - Emergency/Trauma/ICU detection")
print("  2. has_surgery (boolean) - Surgical capability detection")
print("  3. has_imaging (boolean) - Radiology/imaging detection")
print("  4. capability_score (int 0-8) - Overall capability index")
print("  5. is_medical_desert_risk (boolean) - Low capability flag")
print("  6. address_region_clean (string) - Normalized region name")

# Compute statistics
emergency_count = final_enhanced.filter(F.col("has_emergency")).count()
surgery_count = final_enhanced.filter(F.col("has_surgery")).count()
imaging_count = final_enhanced.filter(F.col("has_imaging")).count()
desert_risk_count = final_enhanced.filter(F.col("is_medical_desert_risk")).count()
total = final_enhanced.count()

print("\n📊 CAPABILITY DISTRIBUTION:")
print(f"  Facilities with emergency care: {emergency_count} ({100*emergency_count/total:.1f}%)")
print(f"  Facilities with surgery: {surgery_count} ({100*surgery_count/total:.1f}%)")
print(f"  Facilities with imaging: {imaging_count} ({100*imaging_count/total:.1f}%)")
print(f"  Medical desert risk facilities: {desert_risk_count} ({100*desert_risk_count/total:.1f}%)")

print("\n📈 CAPABILITY SCORE DISTRIBUTION:")
final_enhanced.groupBy("capability_score").count().orderBy("capability_score").show()

print("="*70)
print(f"✓ Enhanced silver table saved: {SILVER}")
print(f"  Total records with intelligence fields: {total}")
print("="*70)

In [0]:
SELECT
  *
FROM
  vf_health.ghana.silver_facilities_clean
LIMIT 20

In [0]:
%python
# ============================================================================
# ENHANCED ORGANIZATION CLASSIFICATION
# ============================================================================
# Optional: Improve organization_type classification using professional utilities
# This adds a quality score and refined classification without breaking existing pipeline

from pyspark.sql import functions as F

if HAS_ORG_CLASSIFICATION:
    print("Enhanced organization classification available")
    print("  Benefits:")
    print("  - Better NGO vs Facility distinction")
    print("  - Translation and naming rules")
    print("  - Structured classification guidelines")
    print("\nNote: For actual LLM-based classification, integrate into 04_idp_agent_extraction")
    print("This cell demonstrates where classification could enhance silver data.")
    
    # Add quality score for organization_type based on available fields
    silver_enhanced = spark.table(SILVER).withColumn(
        "org_type_confidence",
        F.when(
            (F.col("organization_type").isNotNull()) & 
            (F.col("organization_type") != "") &
            (F.size(F.coalesce(F.col("phone_numbers"), F.array())) > 0),
            F.lit(0.9)
        ).when(
            (F.col("organization_type").isNotNull()) & (F.col("organization_type") != ""),
            F.lit(0.7)
        ).otherwise(F.lit(0.3))
    )
    
    # Classify based on name patterns (simple heuristic)
    silver_enhanced = silver_enhanced.withColumn(
        "org_category_hint",
        F.when(
            F.lower(F.col("name")).rlike("hospital|medical center|clinic|pharmacy"),
            F.lit("facility")
        ).when(
            F.lower(F.col("name")).rlike("foundation|ngo|charity|aid|relief"),
            F.lit("ngo")
        ).when(
            F.col("missionStatement").isNotNull(),
            F.lit("likely_ngo")
        ).otherwise(F.lit("unknown"))
    )
    
    print("\n✓ Added classification enhancement columns:")
    print("  - org_type_confidence: Quality score for organization_type")
    print("  - org_category_hint: Heuristic-based category hint")
    
    # Show sample
    print("\nSample with enhanced classification:")
    display(silver_enhanced.select(
        "name", "organization_type", "org_category_hint", "org_type_confidence"
    ).limit(10))
    
    # Optional: Write enhanced version (commented out to avoid overwriting)
    # silver_enhanced.write.mode("overwrite").format("delta").saveAsTable(f"{SILVER}_enhanced")
    # print(f"\n✓ Enhanced silver table: {SILVER}_enhanced")
    
else:
    print("⚠️  Professional classification utilities not available")
    print("   Skipping enhanced organization classification")